# 03 · Pattern Validation & Feature Engineering
Goal: turn each EDA hunch into a clear verdict (**supported / partly / not supported**) using numbers, then freeze the list of features the models will use.

All tests use the **training split only**, so the test customers stay untouched for the final model comparison.

**Setup.**

In [1]:
import sys, pathlib, warnings
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
from credit_kg import config as C
import json
from scipy import stats as st
from credit_kg import data as D, features as F, stats as S

**Load data and compute all candidate baseline features** for every customer. Features never look at the target, so computing them for test customers too is safe.

In [2]:
wide, monthly, split = D.load_processed()
wide = wide.merge(split)
feats = F.baseline_features(wide)
assert np.isfinite(feats.drop(columns="customer_id").to_numpy()).all(), "features contain NaN/inf"
tr = wide[wide.split == "train"].merge(feats)
y = tr[C.TARGET]
tr["late_old3"] = (tr[[f"status_m{m}" for m in (1, 2, 3)]] >= 1).sum(axis=1)
tr["relative_request_30k"] = 30000 / tr.pay_median.clip(lower=1)
feats.describe().T.round(3).head(30)

,count,mean,std,min,25%,50%,75%,max
customer_id,30000.0,15000.500,8660.398,1.000,7500.750,15000.500,22500.250,30000.000
util_avg,30000.0,0.373,0.352,0.000,0.030,0.285,0.688,5.364
util_latest,30000.0,0.424,0.411,0.000,0.022,0.314,0.830,6.455
util_std,30000.0,0.089,0.119,0.000,0.013,0.038,0.124,3.862
util_slope,30000.0,0.023,0.063,-0.463,-0.006,0.001,0.038,1.167
high_util_months,30000.0,0.879,1.699,0.000,0.000,0.000,1.000,6.000
inactive_flag,30000.0,0.136,0.342,0.000,0.000,0.000,0.000,1.000
credit_balance_months,30000.0,0.131,0.614,0.000,0.000,0.000,0.000,6.000
coverage_median,30000.0,0.387,0.433,0.000,0.042,0.082,1.000,1.000
coverage_min,30000.0,0.265,0.409,0.000,0.000,0.036,0.328,1.000


**How a hypothesis is judged.** For each one we bucket the customers, compute the default rate per bucket, and check three things:
1. **direction**: the rank correlation between bucket order and default rate (+1 = rates rise steadily, -1 = fall steadily);
2. **trend test p-value** (Cochran-Armitage): is the trend more than noise?
3. **single-feature AUC**: how well this one feature separates defaulters on its own (0.5 = useless).

In [3]:
def judge(name, x, expected, bins=None, q=None, subset=None):
    xs, ys = (x, y) if subset is None else (x[subset], y[subset])
    t = S.default_rate_table(xs, ys, bins=bins, q=q)
    rho = st.spearmanr(np.arange(len(t)), t["default_rate"]).statistic
    p = S.cochran_armitage(t)
    want = 1 if expected == "increasing" else -1
    if p < 0.01 and rho * want >= 0.8:
        verdict = "SUPPORTED"
    elif p < 0.01 and rho * want >= 0.4:
        verdict = "PARTLY (mostly in expected direction)"
    elif p < 0.01 and rho * want <= -0.4:
        verdict = "NOT SUPPORTED (opposite direction)"
    elif p < 0.01:
        verdict = "NOT SUPPORTED (non-monotone shape)"
    else:
        verdict = "NOT SUPPORTED (no clear trend)"
    return {"hypothesis": name, "expected": expected, "observed_shape": S.is_monotone(t["default_rate"]),
            "direction_rho": round(rho, 2), "trend_p": float(f"{p:.2g}"),
            "single_feature_auc": round(S.single_feature_auc(xs, ys), 3),
            "rate_low_bucket": t["default_rate"].iloc[0], "rate_high_bucket": t["default_rate"].iloc[-1],
            "verdict": verdict}, t

**Run all hypotheses.** H1b repeats the utilization test only for customers who actually use their card (average utilization > 10%), because the EDA showed a U-shape.

In [4]:
U = [-np.inf, 0, .1, .3, .5, .7, .9, 1.0, np.inf]
H = [
    ("H1  higher utilization -> more default", tr.util_avg, "increasing", dict(bins=U)),
    ("H1b same, active users only (util > 10%)", tr.util_avg, "increasing", dict(bins=[.1, .3, .5, .7, .9, 1.0, np.inf], subset=tr.util_avg > .1)),
    ("H2  more late months -> more default", tr.late_months, "increasing", {}),
    ("H3  higher payment coverage -> less default", tr.coverage_median, "decreasing", dict(bins=[-np.inf, 0, .05, .1, .2, .5, .99, np.inf])),
    ("H4  rising utilization (slope) -> more default", tr.util_slope, "increasing", dict(q=5)),
    ("H5  rising payments (slope) -> less default", tr.pay_slope_rel, "decreasing", dict(q=5)),
    ("H6  rising bills (slope) -> more default", tr.bill_slope_rel, "increasing", dict(q=5)),
    ("H7  less consistent payments (CV) -> more default", tr.pay_cv, "increasing", dict(q=5)),
    ("H8  higher credit limit -> less default", tr.credit_limit, "decreasing", dict(q=5)),
    ("H9  larger typical payment -> less default", tr.pay_median, "decreasing", dict(q=5)),
    ("H10 30k request is large vs typical payment -> more default", tr.relative_request_30k, "increasing", dict(bins=[0, 1, 3, 10, 30, 100, np.inf])),
]
rows, detail = [], {}
for name, x, exp, kw in H:
    r, t = judge(name, x, exp, **kw)
    rows.append(r); detail[name] = t
verdicts = pd.DataFrame(rows).set_index("hypothesis")
verdicts.to_csv(C.TABLES / "pattern_validation.csv")
pd.concat(detail, names=["hypothesis"]).to_csv(C.TABLES / "pattern_validation_buckets.csv")
verdicts

,expected,observed_shape,direction_rho,trend_p,single_feature_auc,rate_low_bucket,rate_high_bucket,verdict
hypothesis,,,,,,,,
H1 higher utilization -> more default,increasing,non-monotone,0.31,2.300000e-69,0.568,0.3580,0.3504,NOT SUPPORTED (non-monotone shape)
"H1b same, active users only (util > 10%)",increasing,increasing,1.00,2.000000e-63,0.595,0.1602,0.3504,SUPPORTED
H2 more late months -> more default,increasing,increasing,1.00,0.000000e+00,0.729,0.1151,0.6972,SUPPORTED
H3 higher payment coverage -> less default,decreasing,non-monotone,-0.96,1.600000e-79,0.589,0.5772,0.1683,SUPPORTED
H4 rising utilization (slope) -> more default,increasing,non-monotone,-0.70,1.300000e-41,0.554,0.2990,0.2140,NOT SUPPORTED (opposite direction)
H5 rising payments (slope) -> less default,decreasing,non-monotone,-0.30,2.700000e-03,0.512,0.2208,0.2119,NOT SUPPORTED (non-monotone shape)
H6 rising bills (slope) -> more default,increasing,non-monotone,-0.70,5.000000e-41,0.554,0.2985,0.2140,NOT SUPPORTED (opposite direction)
H7 less consistent payments (CV) -> more default,increasing,non-monotone,0.60,3.900000e-05,0.517,0.1915,0.2044,PARTLY (mostly in expected direction)
H8 higher credit limit -> less default,decreasing,decreasing,-1.00,7.900000e-151,0.623,0.3217,0.1308,SUPPORTED


**H11: does *recent* lateness matter beyond the total count?** Among customers with exactly one late month, we compare 'the late month was in Jul–Sep' with 'the late month was in Apr–Jun' using a two-proportion z-test.

In [5]:
one = tr[tr.late_months == 1]
recent, old = one[one.late_recent3 == 1], one[one.late_recent3 == 0]
k = np.array([recent[C.TARGET].sum(), old[C.TARGET].sum()]); n = np.array([len(recent), len(old)])
p_pool = k.sum() / n.sum(); z = (k[0]/n[0] - k[1]/n[1]) / np.sqrt(p_pool * (1 - p_pool) * (1/n[0] + 1/n[1]))
print(f"late recently: {k[0]/n[0]:.3f} (n={n[0]})   late only earlier: {k[1]/n[1]:.3f} (n={n[1]})   z={z:.1f}, p={2*st.norm.sf(abs(z)):.1g}")

late recently: 0.332 (n=2605)   late only earlier: 0.211 (n=967)   z=7.0, p=2e-12


**H12: behavioural deterioration.** We compare customers whose worst status in Jul–Sep is worse than in Apr–Jun (deteriorating), the same (stable), or better (improving).

In [6]:
risk = tr[[f"status_m{m}" for m in C.MONTHS]].clip(lower=0).to_numpy()
delta = risk[:, 3:].max(1) - risk[:, :3].max(1)
trend = pd.Series(np.select([delta > 0, delta < 0], ["deteriorating", "improving"], "stable"), index=tr.index)
S.default_rate_table(trend, y)

,n,defaults,default_rate,ci_low,ci_high
bucket,,,,,
deteriorating,4371,1749,0.4001,0.3857,0.4147
improving,1578,463,0.2934,0.2715,0.3164
stable,18051,3097,0.1716,0.1661,0.1771


**H13: does the *order* of late months add anything beyond the count?** Within each total of late months we compare the longest consecutive late streak. If default rates barely move across columns, streaks add little on top of the count (useful to know before we build graph features).

In [7]:
streak = F._longest_run(tr[[f"status_m{m}" for m in C.MONTHS]].to_numpy() >= 1)
pd.crosstab(tr.late_months, streak, values=y, aggfunc="mean").round(2)

col_0,0,1,2,3,4,5,6
late_months,,,,,,,
0,0.12,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,0.30,NaN,NaN,NaN,NaN,NaN
2,NaN,0.45,0.39,NaN,NaN,NaN,NaN
3,NaN,1.00,0.53,0.49,NaN,NaN,NaN
4,NaN,NaN,0.68,0.70,0.52,NaN,NaN
5,NaN,NaN,NaN,0.57,0.00,0.61,NaN
6,NaN,NaN,NaN,NaN,NaN,NaN,0.7


**Redundancy check.** Pairs of candidate features with Spearman |ρ| > 0.9 carry almost the same information. For each pair we keep the one with the higher single-feature AUC.

In [8]:
cand = F.ENGINEERED
corr = tr[cand].corr(method="spearman").abs()
auc = {c: S.single_feature_auc(tr[c], y) for c in cand}
pairs = [(a, b, round(corr.loc[a, b], 3)) for i, a in enumerate(cand) for b in cand[i + 1:] if corr.loc[a, b] > 0.9]
redundant = pd.DataFrame(pairs, columns=["feature_a", "feature_b", "spearman"])
redundant["drop"] = [a if auc[a] < auc[b] else b for a, b, _ in pairs]
redundant

,feature_a,feature_b,spearman,drop
0,util_avg,util_latest,0.918,util_latest
1,util_slope,bill_slope_rel,0.999,bill_slope_rel
2,late_months,late_recent3,0.906,late_recent3


**Weak-signal check.** Features whose single-feature AUC is below 0.52 are essentially noise on their own. We drop them unless they are needed as a flag for the scenario layer.

In [9]:
auc_tab = pd.Series(auc).sort_values(ascending=False).round(3).to_frame("single_feature_auc")
auc_tab["weak"] = auc_tab.single_feature_auc < 0.52
auc_tab

,single_feature_auc,weak
late_months,0.729,False
late_recent3,0.717,False
max_status,0.712,False
status_latest,0.693,False
coverage_min,0.649,False
zero_pay_months,0.627,False
pay_latest,0.613,False
pay_median,0.601,False
coverage_latest,0.590,False
coverage_median,0.589,False


**Freeze the feature specification.** Model features = credit limit + the 18 raw monthly values + the engineered features that survived the two checks. Demographics are left out on purpose. The list is saved as a file, so notebooks 06–07 cannot quietly use something else.

In [10]:
# Deliberate exceptions: inactive_flag explains the U-shape; late_recent3 carries recency (H11) beyond the count.
keep_flags = {"inactive_flag", "late_recent3"}
dropped_redundant = set(redundant["drop"]) - keep_flags
dropped_weak = set(auc_tab.index[auc_tab.weak]) - keep_flags
engineered_kept = [c for c in cand if c not in dropped_redundant | dropped_weak]
spec = {
    "raw_features": ["credit_limit"] + F.RAW_MONTHLY,
    "engineered_features": engineered_kept,
    "dropped_redundant": sorted(dropped_redundant),
    "dropped_weak": sorted(dropped_weak),
    "excluded_demographics": C.DEMOGRAPHICS,
    "thresholds": {"high_util": C.HIGH_UTIL, "weak_coverage": C.WEAK_COVERAGE, "full_coverage": C.FULL_COVERAGE,
                   "inactive_util": C.INACTIVE_UTIL, "util_rise": C.UTIL_RISE},
}
json.dump(spec, open(C.FEATURES / "feature_spec.json", "w"), indent=2)
print("kept engineered:", engineered_kept)
print("dropped (redundant):", sorted(dropped_redundant))
print("dropped (weak):", sorted(dropped_weak))

kept engineered: ['util_avg', 'util_slope', 'high_util_months', 'inactive_flag', 'coverage_median', 'coverage_min', 'coverage_latest', 'weak_cov_months', 'full_pay_months', 'late_months', 'late_recent3', 'max_status', 'status_latest', 'paid_full_months', 'no_use_months', 'pay_median', 'pay_latest', 'zero_pay_months']
dropped (redundant): ['bill_slope_rel', 'util_latest']
dropped (weak): ['bill_latest', 'credit_balance_months', 'no_balance_months', 'pay_cv', 'pay_slope_rel', 'util_std']


**Write the feature documentation** (`docs/FEATURES.md`): every kept feature with its group, its single-feature AUC and whether it is kept.

In [11]:
groups = {"util": "A. utilization", "high_util": "A. utilization", "inactive": "A. utilization", "credit_balance": "A. utilization",
          "coverage": "B. payment coverage", "weak_cov": "B. payment coverage", "full_pay": "B. payment coverage", "no_balance": "B. payment coverage",
          "late": "C. delinquency", "max_status": "C. delinquency", "status": "C. delinquency", "paid_full": "C. delinquency", "no_use": "C. delinquency",
          "bill": "D. bills", "pay_median": "E. payments", "pay_latest": "E. payments", "pay_slope": "E. payments", "zero_pay": "E. payments", "pay_cv": "F. stability"}
def grp(c): return next(v for k, v in groups.items() if c.startswith(k))
doc = pd.DataFrame({"feature": cand, "group": [grp(c) for c in cand], "single_feature_auc": [round(auc[c], 3) for c in cand],
                    "status": ["kept" if c in engineered_kept else ("dropped: redundant" if c in dropped_redundant else "dropped: weak") for c in cand]})
lines = ["# Feature specification (generated by notebooks/03_pattern_validation_features.ipynb)", "",
         "Baseline = order-insensitive aggregates + latest value + slope. Transition/path features come from Neo4j (notebook 05).",
         "Month m1 = Apr 2005 ... m6 = Sep 2005. Coverage = payment in month t / bill of month t-1, capped at 1.", "",
         doc.to_markdown(index=False) if hasattr(doc, "to_markdown") else doc.to_string(index=False)]
(C.ROOT / "docs" / "FEATURES.md").write_text("\n".join(lines), encoding="utf-8")
doc.sort_values(["group", "feature"])

,feature,group,single_feature_auc,status
6,credit_balance_months,A. utilization,0.512,dropped: weak
4,high_util_months,A. utilization,0.549,kept
5,inactive_flag,A. utilization,0.510,kept
0,util_avg,A. utilization,0.568,kept
1,util_latest,A. utilization,0.554,dropped: redundant
3,util_slope,A. utilization,0.554,kept
2,util_std,A. utilization,0.518,dropped: weak
9,coverage_latest,B. payment coverage,0.590,kept
7,coverage_median,B. payment coverage,0.589,kept
8,coverage_min,B. payment coverage,0.649,kept


**Save the baseline feature table** for all 30,000 customers.

In [12]:
feats.to_parquet(C.FEATURES / "baseline.parquet", index=False)
print(feats.shape)

(30000, 27)


### Verdicts in plain words
Read the `verdicts` table above together with these notes:
* **Supported:** more late months, lower payment coverage, lower credit limit and smaller typical payments all go with more default. Recent lateness matters more than old lateness.
* **Only partly / reshaped:** utilization. Among active users, higher utilization is riskier, but overall the curve is U-shaped because inactive customers are also risky.
* **Not supported:** "rising utilization or rising bills mean deterioration". The data shows the opposite. Payment trend and payment consistency are weak.
* **Relative request size (H10):** a fixed 30k request that is large compared with the customer's typical payment goes with higher default, which supports the senior's idea. Caveat: with a fixed request this is simply the inverse of typical payment, so it restates payment capacity rather than adding new evidence.
* Streak length adds little once we know the total number of late months (H13). We still test transition features from the graph honestly in notebook 07.